In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm

import glob
from processing import *
from classical_estimates import classical_estimates
from fit_pv import *

In [49]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/vlos_/*'))
#files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*'))
#files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*blos_20260[4-7]*'))

files

['/home/ulyanov/data/solo/phi/2026/vlos_/phi-fdt-vlos_20250101T001503_V202609151912C_0541011501.fits',
 '/home/ulyanov/data/solo/phi/2026/vlos_/phi-fdt-vlos_20250101T031503_V202609151913C_0541011504.fits',
 '/home/ulyanov/data/solo/phi/2026/vlos_/phi-fdt-vlos_20250101T061503_V202609151913C_0541011507.fits',
 '/home/ulyanov/data/solo/phi/2026/vlos_/phi-fdt-vlos_20250102T001503_V202609151913C_0541021501.fits',
 '/home/ulyanov/data/solo/phi/2026/vlos_/phi-fdt-vlos_20250102T031503_V202609151913C_0541021504.fits',
 '/home/ulyanov/data/solo/phi/2026/vlos_/phi-fdt-vlos_20250102T061503_V202609151914C_0541021507.fits',
 '/home/ulyanov/data/solo/phi/2026/vlos_/phi-fdt-vlos_20250103T001503_V202609151914C_0541031501.fits',
 '/home/ulyanov/data/solo/phi/2026/vlos_/phi-fdt-vlos_20250103T031503_V202609151914C_0541031504.fits',
 '/home/ulyanov/data/solo/phi/2026/vlos_/phi-fdt-vlos_20250103T061503_V202609151915C_0541031507.fits',
 '/home/ulyanov/data/solo/phi/2026/vlos_/phi-fdt-vlos_20250104T001503_V20

In [50]:
from datetime import datetime
from wavelengths import *

Q = []
dates = []
soops = []
wvlns = []
crlns = []
crlts = []
temperatures = []
velocities = []
contposes = []


for file in files:
    with fits.open(file) as hdul:
        data = hdul[0].data
        header = hdul[0].header

    contpos = header['CONTPOS'] - 1
    temperature = header['FGOV1PT1']
    soop = header['SOOPNAME']
    velocity = header['OBS_VR'] / 1000

    nx, ny = header['NAXIS2'], header['NAXIS1']
    xc, yc = header['CRPIX2'] - 1, header['CRPIX1'] - 1
    rsun = header['RSUN_ARC'] / header['CDELT1']

    xi, yi = np.mgrid[:nx, :ny]
    mu = np.sqrt(1 - ((xi - xc) ** 2 + (yi - yc) ** 2) / rsun ** 2)

    Q += [np.nanmedian(data) / 1000]
    #Q += [np.nanmean(data / mu * mu ** 2) / np.nanmean(mu ** 2)]
    wvlns += [np.delete(read_wavelengths(header), contpos)[2]]
    temperatures += [temperature]
    dates += [datetime.fromisoformat(header['DATE-OBS'])]
    crlns += [header['CRLN_OBS']]
    crlts += [header['CRLT_OBS']]
    soops += [soop]
    contposes += [contpos]
    velocities += [velocity]

Q = np.array(Q)
dates = np.array(dates)
wvlns = np.array(wvlns)
temperatures = np.array(temperatures)
crlns = np.array(crlns)
crlts = np.array(crlts)
soops = np.array(soops)
contposes = np.array(contposes)
velocities = np.array(velocities)

/tmp/ipykernel_92798/3025502630.py:30: RuntimeWarning: invalid value encountered in sqrt
  mu = np.sqrt(1 - ((xi - xc) ** 2 + (yi - yc) ** 2) / rsun ** 2)


In [51]:
temperature_constant = 4.01225e-2 * 0.75
tuning_constant = 3.513e-4
ref_wavelength = 6173.341
T0 = 61

voltages = (wvlns - ref_wavelength - (temperatures - T0) * temperature_constant) / tuning_constant

In [52]:
synoptics = np.any([soops == 'R_FULL_LRES_LCAD_RS-Synoptics-Low',
                    soops == 'R_FULL_LRES_LCAD_RS-Synoptics-High',
                    ], axis=0)

t56 = np.where(np.all([synoptics,
                     np.abs(temperatures - 56) < 1,
                     dates > datetime(2025,1,1),
                     dates < datetime(2027,1,1)], axis=0))[0]

t61 = np.where(np.all([synoptics,
                     np.abs(temperatures - 61) < 1,
                     dates > datetime(2025,1,1),
                     dates < datetime(2027,1,1)], axis=0))[0]

t66 = np.where(np.all([synoptics,
                     np.abs(temperatures - 66) < 1,
                     dates > datetime(2025,1,1),
                     dates < datetime(2027,1,1)], axis=0))[0]

plt.figure(figsize=(10,8))
plt.plot(voltages[t56], Q[t56] / 8.2, '.')
plt.plot(voltages[t61], Q[t61] / 8.2, '.')
plt.plot(voltages[t66], Q[t66] / 8.2, '.')

plt.ylim(0.95, 1.1)
plt.grid(True)
plt.tight_layout()

In [5]:
from scipy.ndimage import gaussian_filter

fig, ax = plt.subplots(figsize=(10,8))
ax1 = ax.twinx()

ax.plot(dates, Q)
#ax.plot(dates, gaussian_filter(Q, 200))
ax1.plot(dates, crlns, '--', color='tab:orange', lw=1)
#ax1.plot(dates, crlts, '--', color='tab:red', lw=1)

ax1.grid(True)

#ax.set_ylim(-3,3)
fig.tight_layout()

In [12]:
dates[-1]

datetime.datetime(2026, 4, 15, 8, 0, 2, 908000)

In [16]:
plt.figure(figsize=(10,8))
plt.plot(crlts, Q / 8200, '.')

In [174]:
plt.figure(figsize=(10,10))
plt.imshow(data, 'hmimag', vmin=-1000, vmax=1000)
plt.tight_layout()

In [92]:
header['CRLN_OBS']

258.15441

In [88]:
header

SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                  -32 / number of bits per data pixel                  
NAXIS   =                    2 / number of data axes                            
NAXIS1  =                  768 / length of data axis 1                          
NAXIS2  =                  768 / length of data axis 2                          
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT   FITS (Flexible Image Transport System) format is defined in 'Astronomy
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
LONGSTRN= 'OGIP 1.0'           / The HEASARC Long String Convention may be used.
COMMENT   This FITS file may contain long string keyword values that are        
COMMENT   continued over multiple keywords.  The HEASARC convention uses the &  
COMMENT   character at the end of each substring which is then continued        
COMMENT   on the next keywor

In [35]:
500 / 10000

0.05

In [53]:
3.5 / 1.035

3.381642512077295